# Implementation of the 1st training pipeline, the "Raw"

No engineering, no scaling, just plug-and-play.

Only post-training log transformation for consistency in the analysis.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib
import logging
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer

In [2]:
# logging.basicConfig(level=logging.INFO)

In [3]:
root = Path.cwd().parent 
data_path = root / "Real Data" / "log2.csv"
df = pd.read_csv(data_path)

In [4]:
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df)

INFO:sdv.metadata.single_table:Detected metadata:
INFO:sdv.metadata.single_table:{
    "METADATA_SPEC_VERSION": "SINGLE_TABLE_V1",
    "columns": {
        "Source Port": {
            "sdtype": "numerical"
        },
        "Destination Port": {
            "sdtype": "numerical"
        },
        "NAT Source Port": {
            "sdtype": "numerical"
        },
        "NAT Destination Port": {
            "sdtype": "numerical"
        },
        "Action": {
            "sdtype": "categorical"
        },
        "Bytes": {
            "sdtype": "numerical"
        },
        "Bytes Sent": {
            "sdtype": "numerical"
        },
        "Bytes Received": {
            "sdtype": "numerical"
        },
        "Packets": {
            "sdtype": "numerical"
        },
        "Elapsed Time (sec)": {
            "sdtype": "numerical"
        },
        "pkts_sent": {
            "sdtype": "numerical"
        },
        "pkts_received": {
            "sdtype": "numerical"
        }

In [5]:
synthesizer = CTGANSynthesizer(
    metadata=metadata,
    epochs=50,
    verbose=True
)

synthesizer.fit(df)
synthesizer.save('RAW.pkl')

C:\Users\Карен Оганян\AppData\Local\Programs\Python\Python311\Lib\site-packages\sdv\single_table\base.py:144: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\Карен Оганян\AppData\Local\Programs\Python\Python311\Lib\site-packages\sdv\single_table\base.py:122: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
INFO:SingleTableSynthesizer:{'EVENT': 'Instance', 'TIMESTAMP': datetime.datetime(2025, 6, 1, 20, 49, 36, 79937), 'SYNTHESIZER CLASS NAME': 'CTGANSynthesizer', 'SYNTHESIZER ID': 'CTGANSynthesizer_1.20.1_20eed39e55d446b0b05b532690cc6e84'}
INFO:SingleTableSynthesizer:{'EVENT': 'Fit', 'TIMESTAMP': datetime.datetime(2025, 6, 1, 20, 49, 36, 79937), 'SYNTHESIZER CLASS NAME': 'CTGANSynthesizer', 'SYNTHESIZER ID': 'CTGANSynthesizer_1.20.1_20eed39e55d446b0b05b532690cc6e84', 'TOTAL NUMBER OF T

In [6]:
synthetic_data = synthesizer.sample(len(df))

INFO:SingleTableSynthesizer:{'EVENT': 'Sample', 'TIMESTAMP': datetime.datetime(2025, 6, 1, 21, 3, 44, 109384), 'SYNTHESIZER CLASS NAME': 'CTGANSynthesizer', 'SYNTHESIZER ID': 'CTGANSynthesizer_1.20.1_20eed39e55d446b0b05b532690cc6e84', 'TOTAL NUMBER OF TABLES': 1, 'TOTAL NUMBER OF ROWS': 65532, 'TOTAL NUMBER OF COLUMNS': 12}


In [7]:
# df['Bytes_log'] = np.log1p(df['Bytes Sent'] + df['Bytes Received'])
df['Bytes Ratio'] = np.log1p(df['Bytes Sent'] / (df['Bytes Received'] + 1e-6))  

In [8]:
df['pkts_log'] = np.log1p(df['pkts_sent'] + df['pkts_received'])
df['pkts_ratio'] = np.log1p(df['pkts_sent'] / (df['pkts_received'] + 1e-6))

In [9]:
# synthetic_data['Bytes_log'] = np.log1p(synthetic_data['Bytes Sent'] + synthetic_data['Bytes Received'])
synthetic_data['Bytes Ratio'] = np.log1p(synthetic_data['Bytes Sent'] / (synthetic_data['Bytes Received'] + 1e-6))  

In [10]:
synthetic_data['pkts_log'] = np.log1p(synthetic_data['pkts_sent'] + synthetic_data['pkts_received'])
synthetic_data['pkts_ratio'] = np.log1p(synthetic_data['pkts_sent'] / (synthetic_data['pkts_received'] + 1e-6))

In [11]:
cols_to_scale = [
    'Bytes',
    'Bytes Sent',
    'Bytes Received',
    'Packets',
    'Elapsed Time (sec)',
    'pkts_sent',
    'pkts_received'
]

df[cols_to_scale] = df[cols_to_scale].apply(lambda x: np.log1p(x))
synthetic_data[cols_to_scale] = synthetic_data[cols_to_scale].apply(lambda x: np.log1p(x))

In [15]:
synthetic_data.to_csv('RAW.csv')

In [16]:
df.to_csv('RAW_REAL.csv')